# SDXL + Canny ControlNet 建筑材质增强

基础材质图负责已有颜色与内容，预计算边缘图通过 SDXL Canny ControlNet 锁定建筑结构。比较 `0.35 / 0.45 / 0.55` 三档重绘强度，模型缓存和结果都写入 Google Drive。

## 1. 挂载 Drive 并设置持久化缓存

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ifc-ai-render')
CACHE_DIR = Path('/content/drive/MyDrive/ifc-ai-render-cache/huggingface')
RESULT_DIR = DRIVE_ROOT / 'results' / 'sdxl_canny'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(CACHE_DIR / 'hub')
print('模型缓存:', CACHE_DIR)
print('实验结果:', RESULT_DIR)

## 2. 确认 GPU 并获取项目

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), '请在 Colab 的运行时设置中启用 GPU'
print('当前 GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess

REPO_URL = 'https://github.com/StrawberryChen/ifc-ai-render.git'
REPO_DIR = Path('/content/ifc-ai-render')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('当前代码版本:')
!git log -1 --oneline

## 3. 安装依赖（安装后若 Colab 提示重启，重启后从第1节重新运行）

In [ ]:
!pip -q install -r requirements-colab.txt

## 4. 查看并调整实验配置

正常情况下只修改下面的 `overrides`。完整默认值保存在 `configs/sdxl_canny_img2img.json`。`control_scale` 越高越服从边缘，`strength` 越高则重绘越明显。

In [ ]:
import json

CONFIG_PATH = REPO_DIR / 'configs' / 'sdxl_canny_img2img.json'
RUN_CONFIG_PATH = Path('/content/sdxl_canny_img2img_run.json')
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))

# 日常实验主要修改这里
overrides = {
    'input_image': str(REPO_DIR / 'outputs' / 'sdcc' / 'sdcc.material.png'),
    'control_image': str(REPO_DIR / 'outputs' / 'sdcc' / 'sdcc.edge.png'),
    'output_directory': str(RESULT_DIR),
    'seed': 42,
    'strengths': [0.35, 0.45, 0.55],
    'steps': 40,
    'guidance_scale': 5.5,
    'control_scale': 0.8,
    'control_start': 0.0,
    'control_end': 0.9,
}
config['input']['image'] = overrides['input_image']
config['controlnet']['image'] = overrides['control_image']
config['output']['directory'] = overrides['output_directory']
config['inference']['seed'] = overrides['seed']
config['inference']['strengths'] = overrides['strengths']
config['inference']['num_inference_steps'] = overrides['steps']
config['inference']['guidance_scale'] = overrides['guidance_scale']
config['controlnet']['conditioning_scale'] = overrides['control_scale']
config['controlnet']['guidance_start'] = overrides['control_start']
config['controlnet']['guidance_end'] = overrides['control_end']
RUN_CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
print(RUN_CONFIG_PATH.read_text(encoding='utf-8'))

## 5. 先校验路径和配置，不下载模型

In [ ]:
!python inference/generate_sdxl_img2img.py \
  --config "$RUN_CONFIG_PATH" \
  --cache-dir "$CACHE_DIR" \
  --validate-only

## 6. 运行三档 Canny 结构控制实验

第一次执行会把新增的 Canny ControlNet 权重下载到同一个 Drive 缓存；已缓存的 SDXL 不会重复下载。A100 40GB 无需开启 CPU offload。

In [ ]:
!python inference/generate_sdxl_img2img.py \
  --config "$RUN_CONFIG_PATH" \
  --cache-dir "$CACHE_DIR"

## 7. 并排查看输入与结果

In [ ]:
from PIL import Image
from IPython.display import display

print('基础材质图')
display(Image.open(config['input']['image']))
print('Canny 条件图')
display(Image.open(config['controlnet']['image']))
for result_path in sorted(RESULT_DIR.glob('sdcc.strength_*.png')):
    print(result_path.name)
    display(Image.open(result_path))
print((RESULT_DIR / 'run_metadata.json').read_text(encoding='utf-8'))

## 8. 检查 Drive 中的模型和结果占用

In [ ]:
!du -sh "$CACHE_DIR" "$RESULT_DIR" 2>/dev/null || true
!find "$RESULT_DIR" -maxdepth 1 -type f -print